# 3rd-view video hand/gripper masking (SAM2)

This notebook:
1. Loads Kinect 3rd-view frames from a processed `.npz`
2. Lets you click **positive points** on the hand/gripper (pick a prompt frame)
3. Runs SAM2 video propagation to segment the hand/gripper and exports:
   - `hand_seg_overlay.mp4`
   - `video_no_hand_bgfill.mp4` (background fill using a reference frame)

All paths are **relative** to `PIXI_PROJECT_ROOT` (if set), otherwise relative to the current working directory.


In [6]:
# --- Setup: load frames + prepare folders (single cell) ---
%matplotlib inline
%matplotlib widget

import os
from pathlib import Path

import cv2
import numpy as np
import imageio.v2 as imageio

# ============== Paths (relative) ==============
root = Path(os.environ.get("PIXI_PROJECT_ROOT", ""))  # "" -> current working dir

npz_name = "episode_20260302_145648_e7f728_processed"
NPZ_PATH = root / "data" / "real" / f"{npz_name}.npz"

# Put your SAM2 checkpoint under your project root (recommended)
CKPT_PATH = root / "sam2_ckpt" / "sam2.1_hiera_large.pt"

# Output folder (relative)
OUT_DIR = root / "data_processed_script" / "segment" / "predict"
VIDEO_DIR = OUT_DIR / "frames"          # where we dump frames as jpg for SAM2
OUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print("NPZ_PATH :", NPZ_PATH)
print("CKPT_PATH:", CKPT_PATH)
print("OUT_DIR  :", OUT_DIR)

assert NPZ_PATH.exists(), f"npz not found: {NPZ_PATH}"
assert CKPT_PATH.exists(), f"ckpt not found: {CKPT_PATH}"

# ============== Helpers ==============
def unwrap_obj(x):
    if isinstance(x, np.ndarray) and x.dtype == object:
        try:
            return x.item()
        except Exception:
            return x.tolist()
    return x

def decode_kinect_frame(frame):
    """
    Compatible with your data format:
    frame is dict-like and contains "data" -> jpeg bytes (np.ndarray uint8 1D / bytes)
    """
    frame = unwrap_obj(frame)

    if isinstance(frame, dict) and "data" in frame:
        buf = frame["data"]
        if isinstance(buf, np.ndarray):
            buf = buf.tobytes()
        elif isinstance(buf, (bytes, bytearray)):
            buf = bytes(buf)
        else:
            buf = bytes(np.array(buf, dtype=np.uint8))

        arr = np.frombuffer(buf, dtype=np.uint8)
        bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if bgr is None:
            raise ValueError("cv2.imdecode failed for this frame.")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        return rgb

    # fallback: already image array
    if isinstance(frame, np.ndarray) and frame.ndim == 3 and frame.shape[-1] == 3:
        if frame.dtype != np.uint8:
            frame = np.clip(frame, 0, 255).astype(np.uint8)
        return frame

    raise TypeError(f"Unsupported kinect frame type: {type(frame)}")

def load_kinect_rgb_frames(npz_path: Path, max_frames=None):
    data = np.load(npz_path, allow_pickle=True)
    if "kinect_rgb" not in data.files:
        raise KeyError(f"'kinect_rgb' not found. Available keys: {list(data.files)}")

    frames_raw = data["kinect_rgb"]
    T = len(frames_raw)
    if max_frames is not None:
        T = min(T, max_frames)

    frames = [decode_kinect_frame(frames_raw[i]) for i in range(T)]
    frames = np.stack(frames, axis=0)  # (T,H,W,3) uint8 RGB
    return frames, data

def logits_to_mask_hw(mask_logit_tensor):
    """
    Convert SAM2 mask logits tensor to (H,W) boolean mask.
    Handles shapes like (H,W), (1,H,W), (T,H,W)->first.
    """
    ml = mask_logit_tensor.detach().cpu().numpy()
    if ml.ndim == 3 and ml.shape[0] == 1:
        ml = ml[0]
    elif ml.ndim == 3:
        ml = ml[0]
    elif ml.ndim != 2:
        raise ValueError(f"Unexpected mask logits shape: {ml.shape}")
    return (ml > 0)

# ============== Load frames ==============
frames, npz_data = load_kinect_rgb_frames(NPZ_PATH)
print("frames:", frames.shape, frames.dtype)

# Background reference (ideally a frame without hand/gripper)
bg_ref = frames[0].copy()

# ============== Dump frames as jpg for SAM2 ==============
# (Overwrite-safe: re-write in place)
for i, img in enumerate(frames):
    imageio.imwrite(VIDEO_DIR / f"{i:05d}.jpg", img)

print("Saved", len(frames), "frames to:", VIDEO_DIR)


NPZ_PATH : /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/data/real/episode_20260302_145648_e7f728_processed.npz
CKPT_PATH: /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/sam2_ckpt/sam2.1_hiera_large.pt
OUT_DIR  : /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/data_processed_script/segment/predict
frames: (310, 720, 1280, 3) uint8
Saved 310 frames to: /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/data_processed_script/segment/predict/frames


In [7]:
# --- Interactive point picking (positive points only) ---
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

idx_slider = widgets.IntSlider(
    min=0, max=len(frames)-1, step=1,
    value=min(10, len(frames)-1),
    description="pick_frame"
)
clear_btn = widgets.Button(description="Clear points")
use_btn = widgets.Button(description="Use this frame")

display(widgets.HBox([idx_slider, clear_btn, use_btn]))

click_state = {"pos": []}  # list[(x,y)]

fig = go.FigureWidget()
fig.update_layout(width=900, height=550, margin=dict(l=10, r=10, t=10, b=10))

img_trace = go.Image(z=frames[idx_slider.value])
pos_scatter = go.Scatter(
    x=[], y=[],
    mode="markers",
    marker=dict(symbol="cross", size=12),
    name="POS"
)
fig.add_trace(img_trace)
fig.add_trace(pos_scatter)

def refresh_points():
    with fig.batch_update():
        fig.data[1].x = [p[0] for p in click_state["pos"]]
        fig.data[1].y = [p[1] for p in click_state["pos"]]

def on_click(trace, points, selector):
    if not points.xs:
        return
    x, y = float(points.xs[0]), float(points.ys[0])
    click_state["pos"].append((x, y))
    refresh_points()

fig.data[0].on_click(on_click)

def on_change_idx(change):
    # change frame -> clear points (more intuitive)
    click_state["pos"].clear()
    with fig.batch_update():
        fig.data[0].z = frames[int(change["new"])]
        
    refresh_points()

idx_slider.observe(on_change_idx, names="value")

def on_clear(_):
    click_state["pos"].clear()
    refresh_points()

clear_btn.on_click(on_clear)

PROMPT_FRAME_IDX = int(idx_slider.value)
POS_POINTS = click_state["pos"]

def on_use(_):
    global PROMPT_FRAME_IDX, POS_POINTS
    PROMPT_FRAME_IDX = int(idx_slider.value)
    POS_POINTS = list(click_state["pos"])
    print("Locked prompt frame:", PROMPT_FRAME_IDX, "| #POS points:", len(POS_POINTS))

use_btn.on_click(on_use)


display(fig)

print("Tip: click on the image to add POS points. When done, press 'Use this frame', then run the next cell.")


FigureWidget({
    'data': [{'type': 'image',
              'uid': 'c29d7a09-972a-4668-8ea1-13f1ae0ab231',
              'z': {'bdata': ('ZGRigYF/jpCNjY+Mj5GOjpCNjI6LkJ' ... 'zGycrEyMnDyMnDyMnDyMnDx8jCxsfB'),
                    'dtype': 'u1',
                    'shape': '720, 1280, 3'}},
             {'marker': {'size': 12, 'symbol': 'cross'},
              'mode': 'markers',
              'name': 'POS',
              'type': 'scatter',
              'uid': '5aeecdc2-d3d5-4408-ba24-3bc2aae5bec2',
              'x': [],
              'y': []}],
    'layout': {'height': 550, 'margin': {'b': 10, 'l': 10, 'r': 10, 't': 10}, 'template': '...', 'width': 900}
})

Tip: click on the image to add POS points. When done, press 'Use this frame', then run the next cell.


In [8]:
# --- Run SAM2 video propagation + export videos ---

import torch
import cv2
import numpy as np

from sam2.build_sam import build_sam2_video_predictor

MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"  # official config path in repo

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

predictor = build_sam2_video_predictor(MODEL_CFG, str(CKPT_PATH), device=device)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
    state = predictor.init_state(str(VIDEO_DIR))
print("State initialized.")

# If you forgot to click "Use this frame", fall back to current slider value / current points
ann_frame_idx = int(globals().get("PROMPT_FRAME_IDX", 0))
POS_POINTS = list(globals().get("POS_POINTS", []))

print("PROMPT_FRAME_IDX =", ann_frame_idx)
print("num POS points   =", len(POS_POINTS))
assert len(POS_POINTS) > 0, "No POS points. Go back and click points, then press 'Use this frame'."

points = np.array(POS_POINTS, dtype=np.float32)      # (K,2)
labels = np.ones((len(POS_POINTS),), dtype=np.int32) # (K,) all positive

HAND_OBJ_ID = 1

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
    _, obj_ids, mask_logits = predictor.add_new_points_or_box(
        state,
        frame_idx=ann_frame_idx,
        obj_id=HAND_OBJ_ID,
        points=points,
        labels=labels,
    )

# Propagate over full video (forward + backward)
masks_all = {}  # f_idx -> (H,W) bool

def collect(gen):
    for f_idx, obj_ids, mask_logits in gen:
        obj_ids_list = [int(x) for x in obj_ids]
        if HAND_OBJ_ID not in obj_ids_list:
            continue
        j = obj_ids_list.index(HAND_OBJ_ID)
        masks_all[int(f_idx)] = logits_to_mask_hw(mask_logits[j])

# 1) forward: ann_frame_idx -> end
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
    collect(predictor.propagate_in_video(state))
print("after forward:", len(masks_all))

# 2) backward: ann_frame_idx -> start
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
    collect(predictor.propagate_in_video(state, reverse=True))
print("after backward:", len(masks_all), "/", len(frames))

if len(masks_all) > 0:
    print("min frame:", min(masks_all.keys()), "max frame:", max(masks_all.keys()))

# ===== Export videos =====
h, w, _ = frames[0].shape
fps = 15

overlay_path = OUT_DIR / "hand_seg_overlay.mp4"
nohand_path  = OUT_DIR / "video_no_hand_bgfill.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer_overlay = cv2.VideoWriter(str(overlay_path), fourcc, fps, (w, h))
writer_nohand  = cv2.VideoWriter(str(nohand_path),  fourcc, fps, (w, h))

# RGB -> BGR for cv2 writer
mask_color = np.array([255, 0, 0], dtype=np.float32)

for i in range(len(frames)):
    img = frames[i].copy()  # RGB
    m = masks_all.get(i, np.zeros((h, w), dtype=bool))

    overlay = img.copy()
    overlay[m] = (0.6 * overlay[m] + 0.4 * mask_color).astype(np.uint8)

    img_nohand = img.copy()
    img_nohand[m] = bg_ref[m]

    writer_overlay.write(cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    writer_nohand.write(cv2.cvtColor(img_nohand, cv2.COLOR_RGB2BGR))

writer_overlay.release()
writer_nohand.release()

print("Saved:")
print(" -", overlay_path)
print(" -", nohand_path)


device: cuda


frame loading (JPEG): 100%|██████████| 310/310 [00:07<00:00, 43.21it/s]
/mnt/WDC10T/tailai_ws/adaptive_compliance_policy/.pixi/envs/train/lib/python3.11/site-packages/sam2/sam2_video_predictor.py:786: UserWarning:

cannot import name '_C' from 'sam2' (/mnt/WDC10T/tailai_ws/adaptive_compliance_policy/.pixi/envs/train/lib/python3.11/site-packages/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).



State initialized.
PROMPT_FRAME_IDX = 195
num POS points   = 4


propagate in video: 100%|██████████| 115/115 [00:08<00:00, 13.25it/s]


after forward: 115


propagate in video: 100%|██████████| 196/196 [00:14<00:00, 13.14it/s]


after backward: 310 / 310
min frame: 0 max frame: 309
Saved:
 - /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/data_processed_script/segment/predict/hand_seg_overlay.mp4
 - /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/data_processed_script/segment/predict/video_no_hand_bgfill.mp4


In [ ]:
# --- BATCH CELL: process all episode_20260227*.npz in data/real/ ---
# output:
# 1) write modified npz (kinect_rgb replaced by nohand JPEG bytes) to data/real_3rd_view/
# 2) save nohand video mp4 to data_processed_script/segment/predict/
# 3) print frame counts for each modality (check alignment)

import os
import time
from pathlib import Path
from typing import Dict, Tuple, Any, List

import numpy as np
import cv2
import imageio.v2 as imageio
import torch
from tqdm.auto import tqdm

from sam2.build_sam import build_sam2_video_predictor

# =========================
# 0) Paths & file list
# =========================
root = Path(os.environ.get("PIXI_PROJECT_ROOT", ""))
src_dir = root / "data" / "real"
# dst_npz_dir = root / "data" / "real_3rd_view"
dst_npz_dir = root / "data" / "real_3rd_view"/"03_02"
# dst_video_dir = root / "data_processed_script" / "segment" / "predict"
dst_video_dir = root / "data_processed_script" / "segment" / "predict"/"03_02"

dst_npz_dir.mkdir(parents=True, exist_ok=True)
dst_video_dir.mkdir(parents=True, exist_ok=True)

npz_paths = sorted(src_dir.glob("episode_20260302*.npz"))
assert len(npz_paths) > 0, f"No npz found: {src_dir}/episode_20260302*.npz"

print("Found npz:", len(npz_paths))
for p in npz_paths[:5]:
    print(" -", p.name)
if len(npz_paths) > 5:
    print(" ...")

# =========================
# 1) Reference episode: MUST be already processed in your notebook
# =========================
# These should exist from your previous cells:
# frames (npz1 frames), masks_all (npz1 masks), PROMPT_FRAME_IDX, POS_POINTS
frames_ref = frames
masks_ref = masks_all
ann_ref = int(globals().get("PROMPT_FRAME_IDX", 0))
pos_points = list(globals().get("POS_POINTS", []))
assert len(pos_points) > 0, "POS_POINTS is empty. Please run point-picking cell and press 'Use this frame'."
assert ann_ref in masks_ref, f"Reference masks_all missing frame {ann_ref}. Please run SAM2 propagation for the first episode."

hand_mask_ref = masks_ref[ann_ref].astype(bool)
img_ref = frames_ref[ann_ref].copy()  # RGB uint8

# cutout
hand_cutout = img_ref.copy()
hand_cutout[~hand_mask_ref] = 0

# bbox (ROI speed-up)
ys, xs = np.where(hand_mask_ref)
y0, y1 = int(ys.min()), int(ys.max()) + 1
x0, x1 = int(xs.min()), int(xs.max()) + 1
bbox = (x0, y0, x1, y1)

print(f"[REF] ann={ann_ref}, #points={len(pos_points)}, bbox={bbox}, ref_frame_shape={img_ref.shape}")

# =========================
# 2) Helpers
# =========================
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

HAND_OBJ_ID = 1
points_np = np.array(pos_points, dtype=np.float32)
labels_np = np.ones((len(pos_points),), dtype=np.int32)

def make_composite(base_rgb: np.ndarray, alpha: float = 0.9) -> np.ndarray:
    """Paste hand_cutout onto base_rgb within bbox (alpha blend only inside ref mask)."""
    x0, y0, x1, y1 = bbox
    base = base_rgb.copy()
    roi = base[y0:y1, x0:x1].astype(np.float32)
    hand_roi = hand_cutout[y0:y1, x0:x1].astype(np.float32)
    m_roi = hand_mask_ref[y0:y1, x0:x1]
    roi[m_roi] = (1 - alpha) * roi[m_roi] + alpha * hand_roi[m_roi]
    base[y0:y1, x0:x1] = roi.astype(np.uint8)
    return base

def downsample_gray(img: np.ndarray, size: Tuple[int,int]=(128, 72)) -> np.ndarray:
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    g = cv2.resize(g, size, interpolation=cv2.INTER_AREA)
    return g.astype(np.float32) / 255.0

ref_feat = downsample_gray(img_ref)

def auto_find_insert_idx(frames2: np.ndarray) -> int:
    """Find the frame in frames2 most similar to reference prompt frame (global search)."""
    best_i, best_score = 0, 1e18
    for i in range(len(frames2)):
        f = downsample_gray(frames2[i])
        # simple L2 distance
        s = float(np.mean((f - ref_feat) ** 2))
        if s < best_score:
            best_score = s
            best_i = i
    return int(best_i)

def encode_rgb_to_jpeg_bytes(rgb: np.ndarray, quality: int = 95) -> bytes:
    """RGB uint8 HWC -> JPEG bytes"""
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    assert ok, "cv2.imencode failed"
    return buf.tobytes()

def print_modality_lengths(npz_data: Dict[str, Any]) -> Dict[str, int]:
    lengths = {}
    for k, v in npz_data.items():
        try:
            lengths[k] = int(len(v))
        except Exception:
            pass
    # pretty print (sorted)
    print("  modality lengths:")
    for k in sorted(lengths.keys()):
        print(f"   - {k}: {lengths[k]}")
    # alignment check: focus on modalities that look like per-frame sequences
    if len(lengths) > 0:
        vals = list(lengths.values())
        aligned = all(x == vals[0] for x in vals)
        print("  aligned lengths?", aligned)
        if not aligned:
            # show which differ from kinect_rgb if exists
            base = lengths.get("kinect_rgb", None)
            if base is not None:
                bad = {k:v for k,v in lengths.items() if v != base}
                if len(bad) > 0:
                    print("  not aligned vs kinect_rgb:", bad)
    return lengths

def export_nohand_video(frames2: np.ndarray, masks2: Dict[int,np.ndarray], bg_ref: np.ndarray, out_mp4: Path, fps: int = 15):
    h, w, _ = frames2[0].shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_mp4), fourcc, fps, (w, h))
    for i in range(len(frames2)):
        img = frames2[i].copy()
        m = masks2.get(i, np.zeros((h, w), dtype=bool))
        img[m] = bg_ref[m]
        writer.write(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    writer.release()

# =========================
# 3) Batch loop
# =========================
tmp_frames_root = root / "data_processed_script" / "tmp_sam2_frames_aug"
tmp_frames_root.mkdir(parents=True, exist_ok=True)

t_batch0 = time.time()
import gc

def _extract_jpeg_payload(x):
    """
    Return a bytes-like or uint8 array that represents JPEG bitstream.
    Handles: bytes/bytearray/memoryview, np.uint8 array, np.void, dict-wrapped payload.
    """
    # already bytes-like
    if isinstance(x, (bytes, bytearray, memoryview)):
        return x

    # numpy array holding compressed bytes
    if isinstance(x, np.ndarray):
        if x.dtype == np.uint8:
            return x
        # object array with a single element sometimes
        if x.dtype == object and x.size == 1:
            return _extract_jpeg_payload(x.item())

    # np.void sometimes appears when loading object arrays
    if isinstance(x, np.void):
        # try treat as bytes
        try:
            return bytes(x)
        except Exception:
            pass

    # dict-wrapped payload (your current case)
    if isinstance(x, dict):
        # common keys that might contain the jpeg payload
        for k in ["data", "jpg", "jpeg", "buffer", "bytes", "image", "img"]:
            if k in x:
                return _extract_jpeg_payload(x[k])
        # sometimes nested like {"img":{"data":...}}
        # if only one key, unwrap it
        if len(x) == 1:
            return _extract_jpeg_payload(next(iter(x.values())))
        raise TypeError(f"dict does not contain recognizable jpeg payload keys: {list(x.keys())}")

    # sometimes it's a numpy scalar object containing bytes/dict
    if hasattr(x, "item"):
        try:
            return _extract_jpeg_payload(x.item())
        except Exception:
            pass

    raise TypeError(f"Unsupported kinect_rgb element type: {type(x)}")


def decode_kinect_elem_to_rgb(elem) -> np.ndarray:
    """Decode one kinect_rgb element (possibly dict-wrapped) into RGB uint8 HWC."""
    payload = _extract_jpeg_payload(elem)

    if isinstance(payload, (bytes, bytearray, memoryview)):
        arr = np.frombuffer(payload, dtype=np.uint8)
    elif isinstance(payload, np.ndarray) and payload.dtype == np.uint8:
        arr = payload
    else:
        raise TypeError(f"Extracted payload is not bytes/uint8-array: {type(payload)}")

    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    assert bgr is not None, "cv2.imdecode failed (payload not valid JPEG?)"
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def pack_back_like(original_elem, new_jpeg_bytes: bytes):
    """
    Pack the new jpeg bytes back into the SAME container type as original_elem.
    - if original is bytes-like -> return bytes
    - if original is np.ndarray(uint8) -> return np.frombuffer(bytes)
    - if original is dict -> write into same key (prefers data/jpg/jpeg/...) and preserve other keys
    """
    # bytes-like container
    if isinstance(original_elem, (bytes, bytearray, memoryview)):
        return new_jpeg_bytes

    # uint8 array container
    if isinstance(original_elem, np.ndarray) and original_elem.dtype == np.uint8:
        return np.frombuffer(new_jpeg_bytes, dtype=np.uint8)

    # np.void container
    if isinstance(original_elem, np.void):
        return new_jpeg_bytes

    # dict container
    if isinstance(original_elem, dict):
        out = dict(original_elem)  # shallow copy preserve meta
        for k in ["data", "jpg", "jpeg", "buffer", "bytes", "image", "img"]:
            if k in out:
                # keep same inner type if possible
                inner = out[k]
                if isinstance(inner, np.ndarray) and inner.dtype == np.uint8:
                    out[k] = np.frombuffer(new_jpeg_bytes, dtype=np.uint8)
                else:
                    out[k] = new_jpeg_bytes
                return out
        # if dict has only one value, overwrite that
        if len(out) == 1:
            k0 = next(iter(out.keys()))
            out[k0] = new_jpeg_bytes
            return out
        # otherwise fallback add a new key
        out["data"] = new_jpeg_bytes
        return out

    # numpy scalar object
    if hasattr(original_elem, "item"):
        try:
            return pack_back_like(original_elem.item(), new_jpeg_bytes)
        except Exception:
            pass

    # fallback
    return new_jpeg_bytes

def write_rgb_jpg(path: Path, rgb: np.ndarray, quality: int = 95):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])

for idx, npz_path in enumerate(npz_paths):
    ep_name = npz_path.stem
    print("\n========================================================")
    print(f"[{idx+1}/{len(npz_paths)}] {ep_name}")

    # --- load npz keys (keep all) ---
    npz_obj = np.load(npz_path, allow_pickle=True)
    data = {k: npz_obj[k] for k in npz_obj.files}
    npz_obj.close()

    lengths = print_modality_lengths(data)
    assert "kinect_rgb" in data, "npz missing key 'kinect_rgb'"

    kinect_bytes = data["kinect_rgb"]
    T2 = int(len(kinect_bytes))
    assert T2 > 0, "Empty kinect_rgb"

    # --- PASS 1: find insert_idx by similarity (stream decode), also get bg_ref2 and best_frame_rgb ---
    best_i, best_score = 0, 1e18
    bg_ref2 = None
    best_frame_rgb = None

    for i in tqdm(range(T2), desc=f"  scan insert_idx ({ep_name})"):
        rgb = decode_kinect_elem_to_rgb(kinect_bytes[i])
        if i == 0:
            bg_ref2 = rgb.copy()
        f = downsample_gray(rgb)  # (72,128) float
        s = float(np.mean((f - ref_feat) ** 2))
        if s < best_score:
            best_score = s
            best_i = i
            best_frame_rgb = rgb.copy()

    insert_idx = int(best_i)
    print(f"  auto insert_idx={insert_idx} (T={T2}, score={best_score:.6f})")
    assert bg_ref2 is not None and best_frame_rgb is not None

    # --- build temp jpg folder for SAM2, stream write (NO frames2_aug stack) ---
    vid_dir = tmp_frames_root / f"{ep_name}_aug"
    vid_dir.mkdir(parents=True, exist_ok=True)
    for f in vid_dir.glob("*.jpg"):
        f.unlink()

    # write (T2+1) jpgs
    for aug_i in tqdm(range(T2 + 1), desc=f"  writing JPGs ({ep_name})"):
        if aug_i == insert_idx:
            rgb = make_composite(best_frame_rgb, alpha=0.9)
        else:
            src_i = aug_i if aug_i < insert_idx else (aug_i - 1)
            rgb = decode_kinect_elem_to_rgb(kinect_bytes[src_i])
        write_rgb_jpg(vid_dir / f"{aug_i:05d}.jpg", rgb, quality=95)

    # --- SAM2 propagate on this episode only ---
    predictor = build_sam2_video_predictor(MODEL_CFG, str(CKPT_PATH), device=device)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
        state = predictor.init_state(str(vid_dir))
        predictor.add_new_points_or_box(
            state,
            frame_idx=insert_idx,
            obj_id=HAND_OBJ_ID,
            points=points_np,
            labels=labels_np,
        )

    masks_aug: Dict[int, np.ndarray] = {}

    def collect(gen, desc):
        for f_idx, obj_ids, mask_logits in tqdm(gen, desc=f"  {desc} ({ep_name})"):
            obj_ids_list = [int(x) for x in obj_ids]
            if HAND_OBJ_ID not in obj_ids_list:
                continue
            j = obj_ids_list.index(HAND_OBJ_ID)
            masks_aug[int(f_idx)] = logits_to_mask_hw(mask_logits[j])

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
        collect(predictor.propagate_in_video(state), "prop forward")
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
        collect(predictor.propagate_in_video(state, reverse=True), "prop backward")

    print(f"  masks_aug: {len(masks_aug)}/{T2+1}")

    # map aug->orig (remove inserted frame)
    masks2: Dict[int, np.ndarray] = {}
    for aug_i, m in masks_aug.items():
        if aug_i == insert_idx:
            continue
        orig_i = aug_i if aug_i < insert_idx else (aug_i - 1)
        if 0 <= orig_i < T2:
            masks2[int(orig_i)] = m

    print(f"  masks2(mapped): {len(masks2)}/{T2}")

    # --- stream generate: (1) write mp4 (nohand) (2) write back kinect_rgb jpeg bytes (NO nohand_frames stack) ---
    out_mp4 = dst_video_dir / f"{ep_name}_nohand.mp4"
    h, w, _ = bg_ref2.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_mp4), fourcc, 15, (w, h))

    new_kinect = np.empty((T2,), dtype=object)

    for i in tqdm(range(T2), desc=f"  nohand+encode ({ep_name})"):
        rgb = decode_kinect_elem_to_rgb(kinect_bytes[i])

        m = masks2.get(i, np.zeros((h, w), dtype=bool))
        rgb[m] = bg_ref2[m]

        # mp4
        writer.write(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

        # jpeg bytes for npz
        jpeg_bytes = encode_rgb_to_jpeg_bytes(rgb, quality=95)
        new_kinect[i] = pack_back_like(kinect_bytes[i], jpeg_bytes)

    writer.release()
    print("  saved nohand mp4:", out_mp4)

    # replace and save new npz
    data["kinect_rgb"] = new_kinect
    out_npz = dst_npz_dir / f"{ep_name}.npz"
    np.savez_compressed(out_npz, **data)
    print("  saved new npz:", out_npz)

    # --- hard cleanup each episode to avoid VSCode/kernel crash ---
    del predictor, state, masks_aug, masks2, new_kinect, data, kinect_bytes, best_frame_rgb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n✅ Batch done. Total time:", f"{time.time()-t_batch0:.1f}s")

In [5]:
# --- REPROCESS ONE NPZ (manual point picking) and overwrite outputs ---

import os, gc, time
from pathlib import Path
import numpy as np
import cv2
import imageio.v2 as imageio
import torch
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display, clear_output
from tqdm.auto import tqdm

from sam2.build_sam import build_sam2_video_predictor

# =========================
# Paths
# =========================
root = Path(os.environ.get("PIXI_PROJECT_ROOT", ""))

src_real_dir = root / "data" / "real"
dst_npz_dir  = root / "data" / "real_3rd_view"
dst_video_dir = root / "data_processed_script" / "segment" / "predict"
dst_npz_dir.mkdir(parents=True, exist_ok=True)
dst_video_dir.mkdir(parents=True, exist_ok=True)

# reference episode for optional fill frame
REF_NAME = "episode_20260227_180233_9d26cc_processed"
REF_PATH = src_real_dir / f"{REF_NAME}.npz"
assert REF_PATH.exists(), f"Reference npz not found: {REF_PATH}"

# =========================
# Robust JPEG decode/encode (same as your batch cell)
# =========================
def _extract_jpeg_payload(x):
    if isinstance(x, (bytes, bytearray, memoryview)):
        return x
    if isinstance(x, np.ndarray):
        if x.dtype == np.uint8:
            return x
        if x.dtype == object and x.size == 1:
            return _extract_jpeg_payload(x.item())
    if isinstance(x, np.void):
        try:
            return bytes(x)
        except Exception:
            pass
    if isinstance(x, dict):
        for k in ["data", "jpg", "jpeg", "buffer", "bytes", "image", "img"]:
            if k in x:
                return _extract_jpeg_payload(x[k])
        if len(x) == 1:
            return _extract_jpeg_payload(next(iter(x.values())))
        raise TypeError(f"dict has no jpeg payload keys: {list(x.keys())}")
    if hasattr(x, "item"):
        try:
            return _extract_jpeg_payload(x.item())
        except Exception:
            pass
    raise TypeError(f"Unsupported element type: {type(x)}")

def decode_kinect_elem_to_rgb(elem) -> np.ndarray:
    payload = _extract_jpeg_payload(elem)
    if isinstance(payload, (bytes, bytearray, memoryview)):
        arr = np.frombuffer(payload, dtype=np.uint8)
    elif isinstance(payload, np.ndarray) and payload.dtype == np.uint8:
        arr = payload
    else:
        raise TypeError(f"payload not bytes/uint8-array: {type(payload)}")
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    assert bgr is not None, "cv2.imdecode failed"
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def encode_rgb_to_jpeg_bytes(rgb: np.ndarray, quality: int = 95) -> bytes:
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    assert ok, "cv2.imencode failed"
    return buf.tobytes()

def pack_back_like(original_elem, new_jpeg_bytes: bytes):
    if isinstance(original_elem, (bytes, bytearray, memoryview)):
        return new_jpeg_bytes
    if isinstance(original_elem, np.ndarray) and original_elem.dtype == np.uint8:
        return np.frombuffer(new_jpeg_bytes, dtype=np.uint8)
    if isinstance(original_elem, np.void):
        return new_jpeg_bytes
    if isinstance(original_elem, dict):
        out = dict(original_elem)
        for k in ["data", "jpg", "jpeg", "buffer", "bytes", "image", "img"]:
            if k in out:
                inner = out[k]
                out[k] = np.frombuffer(new_jpeg_bytes, dtype=np.uint8) if (isinstance(inner, np.ndarray) and inner.dtype == np.uint8) else new_jpeg_bytes
                return out
        if len(out) == 1:
            k0 = next(iter(out.keys()))
            out[k0] = new_jpeg_bytes
            return out
        out["data"] = new_jpeg_bytes
        return out
    if hasattr(original_elem, "item"):
        try:
            return pack_back_like(original_elem.item(), new_jpeg_bytes)
        except Exception:
            pass
    return new_jpeg_bytes

# =========================
# UI: choose target npz (from real_3rd_view) + choose fill source
# =========================
npz_input = widgets.Text(
    value="episode_20260227_152627_165d55_processed.npz",
    description="bad npz",
    layout=widgets.Layout(width="650px")
)

fill_dropdown = widgets.Dropdown(
    options=[
        ("Use THIS npz frame0", "this"),
        (f"Use REF ({REF_NAME}) frame0", "ref"),
    ],
    value="this",
    description="fill bg",
    layout=widgets.Layout(width="450px")
)

load_btn = widgets.Button(description="Load & show picker", button_style="info")
run_btn  = widgets.Button(description="Run reprocess (overwrite)", button_style="danger")
out = widgets.Output()

display(widgets.HBox([npz_input, fill_dropdown, load_btn, run_btn]))
display(out)

# State holders
state_holder = {
    "ep_name": None,
    "src_npz": None,
    "dst_npz": None,
    "dst_mp4": None,
    "data": None,            # dict of all keys from original src npz
    "kinect_elems": None,    # object array from original src npz
    "frames": None,          # RGB frames decoded for UI (only for viewing)
    "bg_ref": None,          # fill bg RGB
    "T": None,
    "H": None,
    "W": None,
    "PROMPT_FRAME_IDX": 0,
    "POS_POINTS": [],
    "fig": None,
    "click_state": {"pos": []},
    "idx_slider": None,
}

def load_target_npz(_=None):
    with out:
        clear_output(wait=True)
        print("Loading...")

    # parse input
    p = npz_input.value.strip()
    # allow full path or just filename
    if p.endswith(".npz"):
        name = Path(p).stem
    else:
        name = Path(p).name
        if name.endswith(".npz"):
            name = Path(name).stem
        else:
            name = name

    # We overwrite outputs in these locations
    dst_npz = dst_npz_dir / f"{name}.npz"
    dst_mp4 = dst_video_dir / f"{name}_nohand.mp4"

    # Reprocess should read from ORIGINAL real/ (not real_3rd_view)
    src_npz = src_real_dir / f"{name}.npz"
    assert src_npz.exists(), f"Original src npz not found in data/real/: {src_npz}"

    # load src npz keys
    npz_obj = np.load(src_npz, allow_pickle=True)
    data = {k: npz_obj[k] for k in npz_obj.files}
    npz_obj.close()
    assert "kinect_rgb" in data, "npz missing key kinect_rgb"

    kinect_elems = data["kinect_rgb"]
    T = int(len(kinect_elems))
    assert T > 0

    # decode frames for UI (yes this uses memory, but single episode is fine)
    frames = []
    for i in tqdm(range(T), desc="Decoding frames for UI"):
        frames.append(decode_kinect_elem_to_rgb(kinect_elems[i]))
    frames = np.stack(frames, axis=0)
    H, W = frames.shape[1], frames.shape[2]

    # choose bg fill frame
    if fill_dropdown.value == "this":
        bg_ref = frames[0].copy()
    else:
        # load ref frame0
        ref_npz = np.load(REF_PATH, allow_pickle=True)
        ref_data = {k: ref_npz[k] for k in ref_npz.files}
        ref_npz.close()
        ref_k = ref_data["kinect_rgb"]
        bg_ref = decode_kinect_elem_to_rgb(ref_k[0])
        assert bg_ref.shape[:2] == (H, W), "ref bg resolution mismatch"

    # store
    state_holder.update({
        "ep_name": name,
        "src_npz": src_npz,
        "dst_npz": dst_npz,
        "dst_mp4": dst_mp4,
        "data": data,
        "kinect_elems": kinect_elems,
        "frames": frames,
        "bg_ref": bg_ref,
        "T": T,
        "H": H,
        "W": W,
        "PROMPT_FRAME_IDX": min(10, T-1),
        "POS_POINTS": [],
        "click_state": {"pos": []},
    })

    # build picker UI (same style as your point picking)
    idx_slider = widgets.IntSlider(
        min=0, max=T-1, step=1,
        value=min(10, T-1),
        description="pick_frame"
    )
    clear_btn = widgets.Button(description="Clear points")
    use_btn = widgets.Button(description="Use this frame", button_style="success")

    fig = go.FigureWidget()
    fig.update_layout(width=900, height=550, margin=dict(l=10, r=10, t=10, b=10))
    img_trace = go.Image(z=frames[idx_slider.value])
    pos_scatter = go.Scatter(
        x=[], y=[],
        mode="markers",
        marker=dict(symbol="cross", size=12),
        name="POS"
    )
    fig.add_trace(img_trace)
    fig.add_trace(pos_scatter)

    def refresh_points():
        with fig.batch_update():
            fig.data[1].x = [p[0] for p in state_holder["click_state"]["pos"]]
            fig.data[1].y = [p[1] for p in state_holder["click_state"]["pos"]]

    def on_click(trace, points, selector):
        if not points.xs:
            return
        x, y = float(points.xs[0]), float(points.ys[0])
        state_holder["click_state"]["pos"].append((x, y))
        refresh_points()

    fig.data[0].on_click(on_click)

    def on_change_idx(change):
        state_holder["click_state"]["pos"].clear()
        with fig.batch_update():
            fig.data[0].z = frames[int(change["new"])]
        refresh_points()

    idx_slider.observe(on_change_idx, names="value")

    def on_clear(_):
        state_holder["click_state"]["pos"].clear()
        refresh_points()

    def on_use(_):
        state_holder["PROMPT_FRAME_IDX"] = int(idx_slider.value)
        state_holder["POS_POINTS"] = list(state_holder["click_state"]["pos"])
        with out:
            print(f"Locked prompt frame: {state_holder['PROMPT_FRAME_IDX']} | #POS points: {len(state_holder['POS_POINTS'])}")

    clear_btn.on_click(on_clear)
    use_btn.on_click(on_use)

    state_holder["idx_slider"] = idx_slider
    state_holder["fig"] = fig

    with out:
        clear_output(wait=True)
        print("Loaded:", name)
        print("src:", src_npz)
        print("will overwrite:")
        print(" -", dst_npz)
        print(" -", dst_mp4)
        print("T,H,W =", T, H, W)
        print("Fill bg source =", fill_dropdown.label)

        display(widgets.HBox([idx_slider, clear_btn, use_btn]))
        display(fig)
        print("Tip: click to add POS points; press 'Use this frame' before running.")

def run_reprocess(_=None):
    with out:
        print("INPUT :", state_holder["src_npz"])
        print("OUTPUT:", state_holder["dst_npz"])
    ep = state_holder["ep_name"]
    assert ep is not None, "Please click 'Load & show picker' first."
    T = state_holder["T"]
    frames = state_holder["frames"]
    bg_ref = state_holder["bg_ref"]
    data = state_holder["data"]
    kinect_elems = state_holder["kinect_elems"]

    ann = int(state_holder["PROMPT_FRAME_IDX"])
    POS = list(state_holder["POS_POINTS"])
    assert len(POS) > 0, "No POS points. Please click points and press 'Use this frame'."

    with out:
        print("\n▶ Running SAM2 propagation...")

    # 1) write temp jpg folder for SAM2 (single episode, no insertion trick needed)
    tmp_dir = root / "data_processed_script" / "tmp_reprocess_frames" / ep
    tmp_dir.mkdir(parents=True, exist_ok=True)
    for f in tmp_dir.glob("*.jpg"):
        f.unlink()

    for i in tqdm(range(T), desc="Writing JPGs for SAM2"):
        # frames are already RGB uint8
        imageio.imwrite(tmp_dir / f"{i:05d}.jpg", frames[i])

    # 2) SAM2 init + prompt + propagate fwd/bwd
    MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    predictor = build_sam2_video_predictor(MODEL_CFG, str(CKPT_PATH), device=device)

    points = np.array(POS, dtype=np.float32)
    labels = np.ones((len(POS),), dtype=np.int32)
    HAND_OBJ_ID = 1

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
        state = predictor.init_state(str(tmp_dir))
        predictor.add_new_points_or_box(
            state,
            frame_idx=ann,
            obj_id=HAND_OBJ_ID,
            points=points,
            labels=labels,
        )

    masks_all_local = {}

    def collect(gen, desc):
        for f_idx, obj_ids, mask_logits in tqdm(gen, desc=desc):
            obj_ids_list = [int(x) for x in obj_ids]
            if HAND_OBJ_ID not in obj_ids_list:
                continue
            j = obj_ids_list.index(HAND_OBJ_ID)
            masks_all_local[int(f_idx)] = logits_to_mask_hw(mask_logits[j])

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
        collect(predictor.propagate_in_video(state), "Propagate forward")
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
        collect(predictor.propagate_in_video(state, reverse=True), "Propagate backward")

    with out:
        print("Masks collected:", len(masks_all_local), "/", T)
        print("▶ Generating nohand + overwriting outputs...")

    # 3) IMPORTANT: use the SAME images SAM2 saw (read back JPGs), then apply masks
    dst_mp4 = state_holder["dst_mp4"]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    # read first jpg to get size
    first = imageio.imread(tmp_dir / f"{0:05d}.jpg")  # RGB
    h, w, _ = first.shape
    writer = cv2.VideoWriter(str(dst_mp4), fourcc, 15, (w, h))

    new_kinect = np.empty((T,), dtype=object)

    for i in tqdm(range(T), desc="Nohand + JPEG encode (aligned)"):
        # read the exact jpg SAM2 used
        rgb = imageio.imread(tmp_dir / f"{i:05d}.jpg")  # RGB uint8

        m = masks_all_local.get(i, np.zeros((h, w), dtype=bool))
        rgb[m] = bg_ref[m]

        writer.write(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

        jpeg_bytes = encode_rgb_to_jpeg_bytes(rgb, quality=95)
        new_kinect[i] = pack_back_like(kinect_elems[i], jpeg_bytes)

    writer.release()

    data["kinect_rgb"] = new_kinect
    dst_npz = state_holder["dst_npz"]
    np.savez_compressed(dst_npz, **data)

    with out:
        print("✅ Overwritten:")
        print(" -", dst_npz)
        print(" -", dst_mp4)

    # cleanup
    del predictor, state, masks_all_local, new_kinect
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

load_btn.on_click(load_target_npz)
run_btn.on_click(run_reprocess)

Output()

Decoding frames for UI:   0%|          | 0/196 [00:00<?, ?it/s]

Writing JPGs for SAM2:   0%|          | 0/196 [00:00<?, ?it/s]

Propagate forward: 0it [00:00, ?it/s]

Propagate backward: 0it [00:00, ?it/s]

...

Decoding frames for UI:   0%|          | 0/207 [00:00<?, ?it/s]

Writing JPGs for SAM2:   0%|          | 0/207 [00:00<?, ?it/s]

Propagate forward: 0it [00:00, ?it/s]

Propagate backward: 0it [00:00, ?it/s]

Nohand + JPEG encode (aligned):   0%|          | 0/207 [00:00<?, ?it/s]